In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ================================
# Step 1: Import Libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# ================================
# Step 2: Load Dataset
# ================================
train_images = pd.read_csv("/kaggle/input/ahcd1/csvTrainImages 13440x1024.csv")
train_labels = pd.read_csv("/kaggle/input/ahcd1/csvTrainLabel 13440x1.csv")

test_images = pd.read_csv("/kaggle/input/ahcd1/csvTestImages 3360x1024.csv")
test_labels = pd.read_csv("/kaggle/input/ahcd1/csvTestLabel 3360x1.csv")

# Labels start from 1 → shift to 0-27
y = train_labels.values.flatten() - 1
y_test = test_labels.values.flatten() - 1

# Normalize (0-255 → 0-1)
X_full = train_images.values.astype("float32") / 255.0
X_test = test_images.values.astype("float32") / 255.0

# Split Train → Train + Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y, test_size=0.1, random_state=42, stratify=y
)

print("Training data shape:", X_train.shape)   # (12096, 1024)
print("Validation data shape:", X_val.shape)   # (1344, 1024)
print("Test data shape:", X_test.shape)        # (3360, 1024)

In [ ]:
# ================================
# Step 3: Reshape Data
# ================================
# For CNN → (32, 32, 1)
X_train_cnn = X_train.reshape(-1, 32, 32, 1)
X_val_cnn   = X_val.reshape(-1, 32, 32, 1)
X_test_cnn  = X_test.reshape(-1, 32, 32, 1)

# For DNN → Keep Flattened (n, 1024)
X_train_dnn = X_train
X_val_dnn   = X_val
X_test_dnn  = X_test

# Model 1


In [ ]:
# ================================
# Step 4: Build CNN Model
# ================================
model_cnn1 = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32, 32, 1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(28, activation='softmax')
])

In [ ]:
model_cnn1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model 2

In [ ]:
# ================================
# Step 5: Build DNN Model
# ================================
model_dnn = models.Sequential([
    layers.Input(shape=(1024,)),   # بدل Flatten
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(28, activation='softmax')
])

In [ ]:
model_dnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ================================
# Step 6: Train Models
# ================================
history_cnn = model_cnn1.fit(
    X_train_cnn, y_train,
    epochs=20, batch_size=128,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

history_dnn = model_dnn.fit(
    X_train_dnn, y_train,
    epochs=20, batch_size=128,
    validation_data=(X_val_dnn, y_val),
    verbose=1
)

In [ ]:
# ================================
# Step 7: Evaluate on Test Set
# ================================
test_loss_cnn, test_acc_cnn = model_cnn1.evaluate(X_test_cnn, y_test, verbose=0)
test_loss_dnn, test_acc_dnn = model_dnn.evaluate(X_test_dnn, y_test, verbose=0)

print("✅ CNN Test Accuracy:", test_acc_cnn)
print("✅ DNN Test Accuracy:", test_acc_dnn)

In [ ]:
# ================================
# Step 8: Plot Training Curves
# ================================
def plot_history(history, title):
    # Accuracy
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(title + " - Accuracy")
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

    # Loss
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(title + " - Loss")
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


# CNN Plots
plot_history(history_cnn, "CNN Model")

# DNN Plots
plot_history(history_dnn, "DNN Model")

In [ ]:
# ================================
# Visualization of Predictions
# ================================
import matplotlib.pyplot as plt
import numpy as np

numeric_arabic = {
    1: "ا", 2: "ب", 3: "ت", 4: "ث", 5: "ج",
    6: "ح", 7: "خ", 8: "د", 9: "ذ", 10: "ر",
    11: "ز", 12: "س", 13: "ش", 14: "ص", 15: "ض",
    16: "ط", 17: "ظ", 18: "ع", 19: "غ", 20: "ف",
    21: "ق", 22: "ك", 23: "ل", 24: "م", 25: "ن",
    26: "هـ", 27: "و", 28: "ي"
}

n = 100
shift = 100
rows = int(np.ceil(np.sqrt(n)))
cols = int(np.ceil(n / rows))

predictions = model_cnn1.predict(X_test_cnn)   # أو model_dnn + X_test_dnn
predictions_Test = np.argmax(predictions, axis=1)
y_true = y_test   # labels جاهزة

plt.figure(figsize=(cols*1, rows*1))
plt.suptitle('Predicted Arabic Characters', fontsize=16)

for i in range(n):
    plt.subplot(rows, cols, i+1)
    plt.xticks([])
    plt.yticks([])

    # العنوان: الحرف المتوقع
    plt.title(numeric_arabic[predictions_Test[i+shift]+1], fontsize=10)

    # لو التنبؤ غلط → صورة بالـ hot
    if y_true[i+shift] != predictions_Test[i+shift]:
        plt.imshow(X_test_cnn[i+shift].reshape(32,32), cmap="hot")
    else:
        plt.imshow(X_test_cnn[i+shift].reshape(32,32), cmap="gray")

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()